In [2]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gseapy as gp
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Paths set up")

Paths set up


In [3]:
# ----------------------------
# Cell 2 — Load annotated objects
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated.h5ad")

# Re-add cell type labels
cluster_labels_1 = {
    "0": "T cells (resting)",
    "1": "T cells (naive/memory)",
    "2": "NK/Cytotoxic T cells",
    "3": "Activated T cells",
    "4": "Macrophages",
    "5": "Monocytes/DC"
}

cluster_labels_2 = {
    "0": "Endothelial cells",
    "1": "Endothelial cells",
    "2": "CAFs",
    "3": "PVL",
    "4": "Basal epithelial",
    "5": "B cells",
    "6": "Cycling cells",
    "7": "Plasma cells",
    "8": "Cycling epithelial",
    "9": "CD8 T cells",
    "10": "NK cells",
    "11": "T cells",
    "12": "Naive/memory T cells",
    "13": "Luminal epithelial",
    "14": "Macrophages",
    "15": "Monocytes/DC",
    "16": "Cycling myeloid",
    "17": "pDC",
    "18": "Luminal epithelial",
    "19": "Luminal epithelial",
    "20": "Epithelial",
    "21": "Epithelial",
    "22": "Luminal epithelial",
    "23": "Luminal epithelial",
    "24": "Luminal epithelial",
    "25": "Luminal epithelial"
}

adata1.obs["cell_type"] = adata1.obs["leiden_0.8"].map(cluster_labels_1)
adata2.obs["cell_type"] = adata2.obs["leiden_0.6"].map(cluster_labels_2)

print(adata1)
print(adata2)
print("\nGSE114725 tissue types:", adata1.obs["tissue"].unique().tolist())
print("GSE176078 subtypes:", adata2.obs["subtype"].unique().tolist())

AnnData object with n_obs × n_vars = 44662 × 2000
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'leiden_1.0', 'celltypist', 'cell_type'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'celltypist_colors', 'hvg', 'leiden_0.2', 'leiden_0.2_colors', 'leiden_0.4', 'leiden_0.4_colors', 'leiden_0.6', 'leiden_0.6_colors', 'leiden_0.8', 'leiden_0.8_colors', 'leiden_1.0', 'leiden_1.0_colors', 'log1p', 'neighbors', 'patient_colors', 'pca', 'rank_genes_leiden_0.8', 'tissue_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
AnnData object with n_obs × n_vars = 91425 × 2000
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 

In [7]:
from scipy.sparse import issparse
import anndata as ad

def get_raw_counts_for_celltype(adata, cell_type, cell_type_col="cell_type"):
    """Extract raw normalised counts for a specific cell type."""
    
    # Get boolean mask as numpy array
    mask = (adata.obs[cell_type_col] == cell_type).values
    
    # Subset raw matrix directly
    X_raw = adata.raw.X[mask]
    
    if issparse(X_raw):
        X_raw = X_raw.toarray()
    
    # Create small AnnData for just this cell type
    adata_ct = ad.AnnData(
        X=X_raw,
        obs=adata.obs[mask].copy(),
        var=adata.raw.var.copy()
    )
    
    print(f"{cell_type}: {adata_ct.n_obs} cells x {adata_ct.n_vars} genes")
    print(f"Sample values: {X_raw[0, :5]}")
    
    return adata_ct

# Test on T cells from GSE114725
adata1_tcells_raw = get_raw_counts_for_celltype(adata1, "T cells (resting)")

T cells (resting): 11734 cells x 14800 genes
Sample values: [0. 0. 0. 0. 0.]
